# Lab 07-02 — Answer Faithfulness and Correctness

**Track 07 · Evaluation** — retrieval metrics told us *which documents* come back; this lab measures *what the RAG answer does with them*.

An answer has two independent qualities. **Faithfulness**: are the claims in the answer supported by the retrieved context? **Correctness**: does the answer actually match the gold reference? This lab scores a real retrieve→generate→evaluate pipeline on the rag-mini-wikipedia corpus with three metrics — an LLM-judge **faithfulness** score, reference **containment**, and answer↔reference **cosine** similarity.

```text
rag-mini passages (3,200)
  -> BGE embed + FAISS index
  -> retrieve top-3 per question (15 questions)
  -> GroqLLM generates the answer
  -> score: faithfulness (judge) + containment + cosine (reference)
  -> verification gate (--verify)
```


## Setup

This notebook mirrors `curriculum/07-evaluation/02-faithfulness-correctness.py` exactly — the same verified code, split into cells. You can run it from anywhere: the imports cell walks up to the repo root and cd's into it, so every `Data/...` path resolves just like the lab script.

From the terminal, the lab runs as:

```bash
python curriculum/07-evaluation/02-faithfulness-correctness.py          # run + demo
python curriculum/07-evaluation/02-faithfulness-correctness.py --verify # verification gate
```

**LLM keys**: this lab generates answers with GroqLLM and judges with the local Ollama judge — `GROQ_API_KEY` must be in the repo-root `.env` (the imports cell loads it).

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the generator, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

import pandas as pd  # noqa: E402
from dotenv import load_dotenv  # noqa: E402
from embeddings.bge import BGEEmbedding  # noqa: E402
from evaluation.judge import LLMJudge  # noqa: E402
from evaluation.metrics import FaithfulnessMetric  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1. Configuration

The evaluation question: *given a retrieved context, is the answer grounded (faithful) and right (correct)?* The knobs — 15 questions sampled from rag-mini's test set, `top_k = 3` context chunks, the BGE embedder, and a `COSINE_THRESHOLD` for the reference rater.


In [ ]:
# 1. Configuration
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
TEST_PATH = RAG_MINI / "test.parquet"
N_SAMPLE = 15  # judge-evaluated questions (Ollama is local and slow)
TOP_K = 3  # context chunks fed to the generator
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"


## 2. Load — passages + test QA

Two parquet files from rag-mini-wikipedia. Note the passages file has a **single `passage` column** — the loader reads it as plain text (no title/text fields to concatenate). The gold answers in `test.parquet` are terse — often just "yes" or "no" — which is why the correctness metrics below use *containment* and *cosine* instead of brittle exact-match.


In [ ]:
# 2. Load — passages + test QA
def load_passages(path: Path) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for every passage."""
    df = pd.read_parquet(path)
    texts = [str(row["passage"]).strip() for _, row in df.iterrows()]
    ids = [str(i) for i in range(len(df))]
    return texts, ids


def load_test_qa(path: Path) -> list[dict]:
    """Return [{"question": ..., "answer": ...}] from test.parquet."""
    df = pd.read_parquet(path)
    return [{"question": r["question"], "answer": r["answer"]}
            for _, r in df.iterrows()]


## 3. Reference-based correctness (no LLM — deterministic)

The cheap, deterministic side of the evaluation: **cosine** between the answer embedding and the gold-reference embedding, and **containment** — is the gold text present in the answer? These need no LLM, are instant, and are perfectly reproducible. They are crude (they ignore wording that means the same thing), which is exactly why the next section adds an LLM judge on top.


In [ ]:
# 3. Reference-based correctness (no LLM — deterministic)
def normalize(text: str) -> str:
    """Lowercase, strip punctuation/articles and whitespace."""
    cleaned = "".join(c.lower() for c in text if c.isalnum() or c.isspace())
    words = [w for w in cleaned.split() if w not in ("a", "an", "the")]
    return " ".join(words)


def reference_contained(answer: str, reference: str) -> bool:
    """True when the normalized gold answer appears inside the answer.

    The rag-mini gold answers are terse ("yes", "no", a number) while the
    generator is asked to elaborate, so exact equality can never hold.
    Containment is the exact-style check that survives elaboration: the
    gold's normalized words must appear in the answer's normalized words.
    """
    return normalize(reference) in normalize(answer)


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 4. Experiment — retrieve, generate, score

The full pipeline: embed all 3,200 passages, index them in FAISS, retrieve the top-3 context per question, let GroqLLM generate an answer, then score it three ways — **faithfulness** (an LLM judge splits the answer into claims and checks each claim against the context), **containment**, and **cosine**.

Note `device="cpu"` on the embedder: the local Ollama judge holds the shared GPU (5.6 GiB here), so bulk-embedding on CPU avoids CUDA OOM and leaves the GPU for the judge calls that follow.


In [ ]:
# 4. Experiment — retrieve, generate, score
def run_experiment() -> dict:
    passages, passage_ids = load_passages(PASSAGES_PATH)
    test_qa = load_test_qa(TEST_PATH)

    # device="cpu": the local Ollama judge holds the shared GPU (5.6 GiB
    # here); bulk-embedding 3200 passages on CPU avoids CUDA OOM and leaves
    # the GPU for the judge calls that follow.
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device="cpu")
    t0 = time.perf_counter()
    vectors = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(passages, passage_ids)
    ]
    store = FAISSVectorStore(embedding=embedder)
    store.add(chunks, embeddings=vectors)
    retriever = SimilarityRetriever(store, top_k=TOP_K)

    llm = GroqLLM(temperature=0.0)
    judge = LLMJudge()
    faithfulness = FaithfulnessMetric(judge)

    rows: list[dict] = []
    t0 = time.perf_counter()
    for item in test_qa[:N_SAMPLE]:
        question, reference = item["question"], item["answer"]
        context_docs = retriever.retrieve(question)
        context = "\n\n".join(d.page_content for d in context_docs)
        answer = llm.invoke(
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            "Answer in one or two complete sentences, stating the key "
            "fact(s) from the context:"
        ).strip()

        rows.append({
            "question": question,
            "reference": reference,
            "answer": answer,
            "faithfulness": faithfulness.score(question, context, answer),
            "contained": reference_contained(answer, reference),
            "cosine": cosine_similarity(
                judge.embed([answer])[0], judge.embed([reference])[0]
            ),
        })
    llm_s = time.perf_counter() - t0

    def mean(key: str) -> float:
        vals = [r[key] for r in rows]
        return sum(vals) / len(vals) if vals else 0.0

    return {
        "rows": rows,
        "indexed": len(passages),
        "embed_s": embed_s,
        "llm_s": llm_s,
        "metrics": {
            "faithfulness": mean("faithfulness"),
            "containment": mean("contained"),
            "answer_cosine": mean("cosine"),
        },
    }


## 5. Demo

The demo prints the aggregate scores plus three example rows. Watch the *relationship* between faithfulness and containment: an answer can be perfectly faithful to an irrelevant context (high faithfulness, low containment) — the generator faithfully amplified a retrieval miss. That combination is how you localize the fault to retrieval rather than the LLM.


In [ ]:
# 5. Demo — print the artifact
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-02 — Faithfulness and correctness")
    print(f"rag-mini {exp['indexed']} passages, {len(exp['rows'])} questions")
    print("=" * 66)

    m = exp["metrics"]
    print(f"\n[1] Mean scores over {len(exp['rows'])} questions:")
    print(f"    faithfulness (claims supported by context): {m['faithfulness']:.3f}")
    print(f"    reference containment                     : {m['containment']:.3f}")
    print(f"    answer-reference cosine                   : {m['answer_cosine']:.3f}")

    print("\n[2] Three example rows:")
    for row in exp["rows"][:3]:
        print(f"    Q: {row['question'][:60]}")
        print(f"       gold={row['reference'][:40]!r} | "
              f"faith={row['faithfulness']:.2f} | "
              f"contained={row['contained']} | cos={row['cosine']:.2f}")

    print(f"\n[3] Timing: embed {exp['indexed']} passages {exp['embed_s']:.1f}s, "
          f"LLM+judge {len(exp['rows'])} Q {exp['llm_s']:.1f}s")

    print(f"\n[4] Takeaway")
    print("    Faithfulness and correctness are orthogonal: an answer can be")
    print("    perfectly faithful to a context that is itself irrelevant")
    print("    (high faithfulness, low containment), or wrong while fully")
    print("    supported by the evidence (the generator faithfully amplified")
    print("    a retrieval miss). Two operational notes: faithfulness needs")
    print("    claim-rich answers — a terse 'yes' has zero claims and scores")
    print("    0.0 by construction — and short-answer gold sets force you to")
    print("    soften exact-match into containment or cosine. High faithfulness")
    print("    + low correctness points at retrieval, not the LLM.")


## 6. Verification gate

The `--verify` gate checks the scores are in range and that faithfulness is actually positive — elaborated answers carry claims; a terse "yes" has zero claims and scores 0.0 by construction, which is why the prompt asks for complete sentences.


In [ ]:
# 6. Verification gate — run ``python <lab> --verify`` from the repo root
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    m = exp["metrics"]
    n = len(exp["rows"])

    checks.append((f"{N_SAMPLE} questions evaluated (>= 10)", n >= 10))
    checks.append(("faithfulness mean in [0, 1]", 0.0 <= m["faithfulness"] <= 1.0))
    checks.append(("faithfulness > 0 (elaborated answers carry claims)",
                   m["faithfulness"] > 0.0))
    checks.append(("reference containment in [0, 1]",
                   0.0 <= m["containment"] <= 1.0))
    checks.append(("answer cosine in [-1, 1]", -1.0 <= m["answer_cosine"] <= 1.0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Bulk-embedding 3,200 passages on CPU takes a few minutes; the 15 LLM generations + judge calls follow. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces.


In [ ]:
verify_gate(exp)
